# Bounded-alpha single-shot timing baseline

Fresh wall-clock baseline for the original bounded-alpha identifier. This executes the committed August broad single-shot benchmark unchanged scientifically, while using a new short cache/results namespace `baseline_b_ss` so no historical cache can be reused.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

HERE = Path.cwd()

BASE_CANDIDATES = [
    HERE / "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed_BACKFILL_FIXED.ipynb",
    HERE / "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed.ipynb",
]
BASE_NOTEBOOK = next((p for p in BASE_CANDIDATES if p.exists()), None)
if BASE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Could not find the committed broad single-shot benchmark. Looked for:\n"
        + "\n".join(str(p) for p in BASE_CANDIDATES)
    )

print("Bounded-alpha timing baseline base:", BASE_NOTEBOOK.name)

base_nb = json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))
patched_study = False
patched_results = False
verified_bounded_identifier = False
patched_cells: list[tuple[int, str]] = []

for cell_index, cell in enumerate(base_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if "identify_nonlinear_unbounded" in src:
        raise RuntimeError(
            "Base notebook unexpectedly references identify_nonlinear_unbounded; "
            "refusing to use it as the bounded timing baseline."
        )

    if (
        "opinion_dynamics.experiments.online_single_shot" in src
        or "opinion_dynamics.identify_nonlinear" in src
    ):
        verified_bounded_identifier = True

    new_src, n = re.subn(
        r'STUDY_NAME\s*=\s*"[^"]+"',
        'STUDY_NAME = "baseline_b_ss"',
        src,
        count=1,
    )
    if n:
        src = new_src
        patched_study = True

    new_src, _ = re.subn(
        r'PIPELINE_VERSION\s*=\s*"[^"]+"',
        'PIPELINE_VERSION = "2026-09-06-bounded-timing-baseline-v1"',
        src,
        count=1,
    )
    src = new_src

    for old_name in (
        "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed_BACKFILL_FIXED",
        "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed",
    ):
        if old_name in src:
            src = src.replace(old_name, "baseline_b_ss")
            patched_results = True

    patched_cells.append((cell_index, src))

required = {
    "bounded/original benchmark import path": verified_bounded_identifier,
    "fresh short STUDY_NAME": patched_study,
    "fresh short results namespace": patched_results,
}
missing = [name for name, ok in required.items() if not ok]
if missing:
    raise RuntimeError(
        "Refusing to execute timing baseline because expected base structure "
        "was not found. Missing: " + ", ".join(missing)
    )

print("Bounded timing-baseline overrides validated:")
for name in required:
    print("  OK:", name)

g = globals()
for cell_index, src in patched_cells:
    print(f"[base code cell {cell_index}]")
    exec(
        compile(src, f"{BASE_NOTEBOOK.name}:cell_{cell_index}", "exec"),
        g,
        g,
    )
